<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Module_15_clean_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [35]:
!pip install torchinfo
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
from torchinfo import summary

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# Train test split

In [8]:
X = df.drop(['id', 'diagnosis', 'Unnamed: 32'], axis = 1)
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scaling and Encoding

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Numpy array to tensor

In [16]:
X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.float32)

# Dataset and Dataloader

In [20]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

# Dataset

In [21]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

# Loader

In [22]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

# Defining the model

In [24]:
class MyNNModel(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)

    return out

# Defining Parameters

In [25]:
learning_rate = 0.1
epochs = 25
criterion = nn.BCELoss()

# Training Pipeline

In [28]:
model = MyNNModel(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

batch_size = 32
n_sample = len(X_train_tensor)

for epoch in range(epochs):
  for start_idx in range(0, n_sample, batch_size):
    end_idx = start_idx + batch_size

    X_batch = X_train_tensor[start_idx:end_idx]
    y_batch = y_train_tensor[start_idx:end_idx]

    # Forward pass
    y_pred = model(X_batch)

    # Loss function
    loss = criterion(y_pred, y_batch.reshape(-1, 1))

    # backpropagation
    loss.backward()

    # optimizer step
    optimizer.step()

    # zero grad
    optimizer.zero_grad()

    print(f"Epoch:{epoch+1}/{epoch} | Loss: {loss.item()} ")

Epoch:1/0 | Loss: 0.6674045324325562 
Epoch:1/0 | Loss: 0.42155569791793823 
Epoch:1/0 | Loss: 0.39750999212265015 
Epoch:1/0 | Loss: 0.3297119736671448 
Epoch:1/0 | Loss: 0.3428739309310913 
Epoch:1/0 | Loss: 0.3280620574951172 
Epoch:1/0 | Loss: 0.3108566701412201 
Epoch:1/0 | Loss: 0.2455717921257019 
Epoch:1/0 | Loss: 0.2626775801181793 
Epoch:1/0 | Loss: 0.20176956057548523 
Epoch:1/0 | Loss: 0.23945419490337372 
Epoch:1/0 | Loss: 0.3225412368774414 
Epoch:1/0 | Loss: 0.22048497200012207 
Epoch:1/0 | Loss: 0.2177003026008606 
Epoch:1/0 | Loss: 0.2116103619337082 
Epoch:2/1 | Loss: 0.16097988188266754 
Epoch:2/1 | Loss: 0.21351048350334167 
Epoch:2/1 | Loss: 0.1842239797115326 
Epoch:2/1 | Loss: 0.12382400780916214 
Epoch:2/1 | Loss: 0.2045188546180725 
Epoch:2/1 | Loss: 0.18840429186820984 
Epoch:2/1 | Loss: 0.23639307916164398 
Epoch:2/1 | Loss: 0.1468936651945114 
Epoch:2/1 | Loss: 0.16199174523353577 
Epoch:2/1 | Loss: 0.1211499273777008 
Epoch:2/1 | Loss: 0.16871990263462067 


# Model evaluation

In [34]:
model.eval()
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features)
    y_pred = (y_pred > 0.5).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list) / len(accuracy_list)

print(f"Overall accuracy: {round(overall_accuracy, 4)}")

Overall accuracy: 0.9783


# Summary

In [37]:
summary(model, X_train_tensor.shape)

Layer (type:depth-idx)                   Output Shape              Param #
MyNNModel                                [455, 1]                  --
├─Linear: 1-1                            [455, 1]                  31
├─Sigmoid: 1-2                           [455, 1]                  --
Total params: 31
Trainable params: 31
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.05
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.06